In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = sns.load_dataset("tips")
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
df.time.unique()

['Dinner', 'Lunch']
Categories (2, object): ['Lunch', 'Dinner']

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [5]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
df['time']=encoder.fit_transform(df['time'])

In [6]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,0,2
1,10.34,1.66,Male,No,Sun,0,3
2,21.01,3.50,Male,No,Sun,0,3
3,23.68,3.31,Male,No,Sun,0,2
4,24.59,3.61,Female,No,Sun,0,4


In [7]:
df.time.unique()


array([0, 1])

In [8]:
x=df.drop('time',axis=1)
y=df['time']

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=1)

In [11]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [12]:
df.dtypes

total_bill     float64
tip            float64
sex           category
smoker        category
day           category
time             int32
size             int64
dtype: object

In [29]:
cat=[]
num=[]
for i in x.columns:
    if df[i].dtype=='category':
        cat.append(i)
    else:
        num.append(i)

In [30]:
cat

['sex', 'smoker', 'day']

In [31]:
df.columns

Index(['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size'], dtype='object')

In [32]:
num

['total_bill', 'tip', 'size']

In [33]:
num_pipeline=Pipeline(steps=[('imputation',SimpleImputer(strategy='median')),('scaling',StandardScaler())])
cat_pipeline=Pipeline(steps=[('imputation',SimpleImputer(strategy='most_frequent')),('encoding',OneHotEncoder())])

In [34]:
preprocessor=ColumnTransformer([("num_pipeline",num_pipeline,num),("cat_pipeline",cat_pipeline,cat)])

In [37]:
x_train_p=preprocessor.fit_transform(X_train,y_train)

In [38]:
x_test_p=preprocessor.transform(X_test)

In [39]:
x_train_p

array([[-0.28611937, -1.47443803, -0.57766863, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.02695905, -0.71612531,  1.47042924, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.3716196 ,  1.19880579,  1.47042924, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.23206267,  0.43283335, -0.57766863, ...,  0.        ,
         0.        ,  1.        ],
       [-1.06543688, -1.29060464, -0.57766863, ...,  1.        ,
         0.        ,  0.        ],
       [-0.29287646,  0.1034652 ,  0.44638031, ...,  1.        ,
         0.        ,  0.        ]])

In [40]:
x_test_p

array([[-1.85376383, -1.48209775, -1.60171757,  1.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [-0.08453291,  0.04984713, -0.57766863,  1.        ,  0.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.79501474,  0.36389583,  0.44638031,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [-0.59356688, -0.33313909, -0.57766863,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.18349826,  0.04984713, -0.57766863,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [-1.32783714, -1.14506988, -0.57766863,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.   

In [41]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [43]:
models = {"support vector classifier": SVC(),
         "DT classifier": DecisionTreeClassifier(),
          'Lg':LogisticRegression()}

In [44]:
from sklearn.metrics import *

In [51]:
def model_eval(x_train_p,y_train,x_test_p,y_test,models):
    evaluation={}
    for i in range(len(models)):
        model=list(models.values())[i]
        model.fit(x_train_p,y_train)
        y_pred=model.predict(x_test_p)
        model_score=accuracy_score(y_test,y_pred)
        evaluation[list(models.keys())[i]]=model_score
    return evaluation

In [52]:
model_eval(x_train_p, y_train, x_test_p, y_test, models)

{'support vector classifier': 0.9183673469387755,
 'DT classifier': 0.8979591836734694,
 'Lg': 0.9183673469387755}

In [53]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification


X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)


rf_classifier = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)


rf_classifier.fit(X, y)

oob_score = rf_classifier.oob_score_
print("Out-of-Bag Score:", oob_score)

Out-of-Bag Score: 0.895


In [54]:
from sklearn.ensemble import RandomForestClassifier

In [55]:
rf = RandomForestClassifier()

In [56]:
from sklearn.model_selection import RandomizedSearchCV
params={'max_depth':[1, 2, 3,5,10,None],
        'n_estimators':[50, 100,200,300],
        'criterion':['gini','entropy']}

In [57]:
params

{'max_depth': [1, 2, 3, 5, 10, None],
 'n_estimators': [50, 100, 200, 300],
 'criterion': ['gini', 'entropy']}

In [58]:
clf = RandomizedSearchCV(rf, param_distributions=params, cv = 5, verbose =3, scoring = 'accuracy')

In [59]:
clf.fit(x_train_p,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END criterion=gini, max_depth=3, n_estimators=50;, score=0.923 total time=   0.1s
[CV 2/5] END criterion=gini, max_depth=3, n_estimators=50;, score=0.974 total time=   0.1s
[CV 3/5] END criterion=gini, max_depth=3, n_estimators=50;, score=1.000 total time=   0.1s
[CV 4/5] END criterion=gini, max_depth=3, n_estimators=50;, score=1.000 total time=   0.1s
[CV 5/5] END criterion=gini, max_depth=3, n_estimators=50;, score=1.000 total time=   0.1s
[CV 1/5] END criterion=gini, max_depth=None, n_estimators=300;, score=0.923 total time=   1.2s
[CV 2/5] END criterion=gini, max_depth=None, n_estimators=300;, score=0.949 total time=   1.1s
[CV 3/5] END criterion=gini, max_depth=None, n_estimators=300;, score=1.000 total time=   1.2s
[CV 4/5] END criterion=gini, max_depth=None, n_estimators=300;, score=0.974 total time=   1.1s
[CV 5/5] END criterion=gini, max_depth=None, n_estimators=300;, score=0.974 total time=   0.9s
[CV 1/5] 

,estimator,RandomForestClassifier()
,param_distributions,"{'criterion': ['gini', 'entropy'], 'max_depth': [1, 2, ...], 'n_estimators': [50, 100, ...]}"
,n_iter,10
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [61]:
y_pred=clf.predict(x_test_p)

In [62]:
accuracy_score(y_test,y_pred)

0.9183673469387755

In [63]:
clf.best_params_

{'n_estimators': 50, 'max_depth': 3, 'criterion': 'gini'}